# AirAware AQI Regression — Hyperparameter Optimization

Run this notebook after creating `X_train`, `X_test`, `y_train`, and `y_test`. Randomized search tunes Random Forest and XGBoost separately, then evaluates each on the test set.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 1. Tune Random Forest
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_params = {
    "n_estimators": [200, 300, 500, 700],
    "max_depth": [None, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 6],
    "max_features": [1.0, "sqrt", 0.8],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_params,
    n_iter=25,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

print("Best Random Forest parameters:")
print(rf_search.best_params_)
print("Best CV MAE:", -rf_search.best_score_)

# 2. Tune XGBoost
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1
)

xgb_params = {
    "n_estimators": [200, 300, 500, 700, 900],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_params,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_

print("\nBest XGBoost parameters:")
print(xgb_search.best_params_)
print("Best CV MAE:", -xgb_search.best_score_)

# 3. Evaluate the tuned individual models on the untouched test set
for name, estimator in [("Tuned Random Forest", best_rf),
                         ("Tuned XGBoost", best_xgb)]:
    pred = estimator.predict(X_test)
    print(f"\n{name}")
    print("MAE:", mean_absolute_error(y_test, pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
    print("R2:", r2_score(y_test, pred))
